In [1]:
# Colab cell (python)
from google.colab import drive
drive.mount('/content/drive')

# clone if not present, else fetch+reset to remote branch
import os
if not os.path.exists('/content/repo'):
    !git clone --depth 1 --branch feat/colab-migration https://github.com/mb6226/autonomous-quant-research-system.git /content/repo
else:
    %cd /content/repo
    !git fetch --all
    !git reset --hard origin/feat/colab-migration
    %cd /content

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/repo
Fetching origin
HEAD is now at 1122ebc Prepare Colab sync
/content


In [ ]:
%%bash
cd /content/repo
pip install -r requirements.txt
pip install --upgrade --force-reinstall numpy==2.4.6 scipy==1.17.1 lightgbm

**Important — safe restart instructions**

- If you are running this notebook in Colab: after installing binary packages (numpy/scipy/lightgbm) restart the runtime from the menu `Runtime -> Restart runtime`.
- If you are running in VS Code / Jupyter: do NOT run `os.kill(os.getpid(), 9)` in a code cell — that will crash the kernel. Use the command palette or the Kernel menu to `Restart Kernel`.
- After restarting, re-run the cell that sets `sys.path` and clears cached `app` modules before importing the repo code. See the next cell for a safe import test.

## Split large 1m.parquet into monthly chunks (Colab)
This cell will: copy `data/raw/EURUSD/1m.parquet` from Drive if needed,
split it into per-month files `data/raw/EURUSD/1m_gapfill_part_YYYY-MM.parquet`, and write
a manifest to both `artifacts/eurusd_split_manifest.json` (repo) and Drive. Run this only once.

In [3]:
from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone
import shutil

REPO_ROOT = Path('/content/repo')
MAIN_PARQUET = REPO_ROOT / 'data/raw/EURUSD/1m.parquet'
DRIVE_SRC = Path('/content/drive/MyDrive/autonomous-quant-research-system/data/raw/EURUSD/1m.parquet')
DRIVE_MANIFEST = Path('/content/drive/MyDrive/autonomous-quant-research-system/artifacts/eurusd_split_manifest.json')
LOCAL_MANIFEST = REPO_ROOT / 'artifacts/eurusd_split_manifest.json'
OUT_DIR = REPO_ROOT / 'data/raw/EURUSD'
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

# If main parquet is not in the repo, try to copy from Drive
if not MAIN_PARQUET.exists():
    if DRIVE_SRC.exists():
        print('Copying main parquet from Drive to repo...')
        shutil.copy2(DRIVE_SRC, MAIN_PARQUET)
    else:
        raise FileNotFoundError(f'Main parquet not found in repo or Drive: {MAIN_PARQUET}')

print('Reading', MAIN_PARQUET)
df = pd.read_parquet(MAIN_PARQUET)
if 'timestamp' not in df.columns:
    raise RuntimeError('No timestamp column in parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df['year_month'] = df['timestamp'].dt.strftime('%Y-%m')

manifest = {}
months = sorted(df['year_month'].unique())
for m in months:
    out_file = OUT_DIR / f'1m_gapfill_part_{m}.parquet'
    sub = df[df['year_month'] == m].drop(columns=['year_month'])
    if len(sub) == 0:
        continue
    print('Writing', out_file, 'rows=', len(sub))
    tmp = out_file.with_suffix('.parquet.tmp')
    sub.to_parquet(tmp, index=False)
    try:
        tmp.replace(out_file)
    except Exception:
        tmp.rename(out_file)
    manifest[m] = {
        'file': str(out_file),
        'rows': int(len(sub)),
        'start_timestamp': sub['timestamp'].min().isoformat(),
        'end_timestamp': sub['timestamp'].max().isoformat()
    }

# write manifest locally and to Drive
with open(LOCAL_MANIFEST, 'w') as f:
    json.dump({'generated_at': datetime.now(timezone.utc).isoformat(), 'chunks': manifest}, f, indent=2)
DRIVE_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
with open(DRIVE_MANIFEST, 'w') as f:
    json.dump({'generated_at': datetime.now(timezone.utc).isoformat(), 'chunks': manifest}, f, indent=2)

print('Split complete. Manifest written to', LOCAL_MANIFEST, 'and', DRIVE_MANIFEST)
print('Chunks produced:', len(manifest))

Reading /content/repo/data/raw/EURUSD/1m.parquet
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-01.parquet rows= 27612
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-02.parquet rows= 27486
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-03.parquet rows= 30804
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-04.parquet rows= 28906
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-05.parquet rows= 29741
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-06.parquet rows= 30297
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-07.parquet rows= 29320
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-08.parquet rows= 29604
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-09.parquet rows= 29104
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-10.parquet rows= 28541
Writing /content/repo/data/raw/EURUSD/1m_gapfill_part_2010-11.parquet rows= 30675
Writing /content/repo/data/raw/EURUSD/1m_gapfill_

In [ ]:
# Import split manifest (from Drive if needed) into the gapfill progress manifest
from pathlib import Path
import json, subprocess, shutil
repo = Path('/content/repo')
progress = repo / 'artifacts/eurusd_gapfill_progress.json'
split = repo / 'artifacts/eurusd_split_manifest.json'
drive_manifest = Path('/content/drive/MyDrive/autonomous-quant-research-system/artifacts/eurusd_split_manifest.json')
# copy manifest from Drive if not present in repo
if not split.exists() and drive_manifest.exists():
    print('Copying split manifest from Drive to repo...')
    shutil.copy2(drive_manifest, split)
if not split.exists():
    print('No split manifest found. Run the split cell first.')
else:
    print('Importing split manifest into gapfill progress...')
    subprocess.run(['python','scripts/import_split_manifest_into_gapfill_progress.py'], cwd=str(repo))
    if progress.exists():
        j = json.loads(progress.read_text())
        print('Progress entries:', len(j.get('progress', [])))
        keys = sorted(j.get('checkpoints', {}).keys())
        print('Recorded months (first 8):', keys[:8])
    else:
        print('Progress file not created.')

In [ ]:
%%bash
cd /content/repo
echo 'Starting gapfill runner (logs -> artifacts/eurusd_gapfill_colab_run.log)'
PYTHONPATH=/content/repo python scripts/dukascopy_gapfill_runner_2025_05_2026_06.py 2>&1 | tee artifacts/eurusd_gapfill_colab_run.log